In [1]:
import pandas as pd
from pathlib import Path
from datetime import datetime
from openpyxl import load_workbook

In [3]:
folder = Path("../tagging 1673")
files = list(folder.glob("*.xlsx"))

if not files:
    raise FileNotFoundError("Tidak ada file Excel di folder tagging 1673")


print(f"Ditemukan {len(files)} file:")
for f in files:
    print(" -", f.name)

# Baca semua file lalu gabungkan
list_df = []
for f in files:
    df = pd.read_excel(f, dtype=str)
    df["source_file"] = (
        f.name
    )  # opsional: kolom penanda asal file, bisa dihapus kalau tidak perlu
    list_df.append(df)

df_gabungan = pd.concat(list_df, ignore_index=True)

print(f"\nTotal baris setelah digabung: {len(df_gabungan)}")
df_gabungan.head()

Ditemukan 6 file:
 - 1.xlsx
 - 2.xlsx
 - 3.xlsx
 - 4.xlsx
 - sqllab_untitled_query_3_20260831T102324.xlsx
 - sqllab_untitled_query_3_20260831T102336.xlsx

Total baris setelah digabung: 53228


,assignment_id,nama_usaha_bang,nama_kk,nama_usaha_bang__1,ada_keluarga_label,geotag_accuracy,geotag_latitude,geotag_longitude,source_file
0,0008551e-7425-436f-a168-274cfe006157,EDITED BY Admin Kabupaten,NURHAYATI,NaN,1. Ditemukan,NaN,-4.0191925,103.2508815,1.xlsx
1,005b1dfc-e6d7-43e2-b261-5eddd654b044,APPROVED BY Pengawas,AQSHAL MADINE,NaN,1. Ditemukan,NaN,-4.01701276,103.25300602,1.xlsx
2,009f0d8c-1e50-45c5-a9bb-48618a528b74,APPROVED BY Pengawas,SUSRIANTO,NaN,1. Ditemukan,NaN,-4.075015,103.3489813,1.xlsx
3,00af1072-4d62-4c06-8fb6-634d5a082723,APPROVED BY Pengawas,NIL NOPRIANI,NaN,1. Ditemukan,NaN,-4.0242062,103.1904475,1.xlsx
4,00bfef5a-a9ba-45c1-8f15-07aa3043d300,APPROVED BY Pengawas,NaN,RUMAH KOSONG RASNI,NaN,2. Baru,-4.076583266666666,103.1772908833333,1.xlsx


In [ ]:
# df_gabungan = df_gabungan.drop(columns=["index1"])

df_gabungan["keberadaan"] = df_gabungan["keberadaan_usaha"].fillna(
    df_gabungan["keberadaan_kk"]
)

group_cols = [
    "assignment_id",
    "nama_usaha_bang",
    "nama_kk",
    "ada_keluarga_label",
    "nama_kk_or_usaha",
    "geotag_accuracy",
    "geotag_latitude",
    "geotag_longitude",
]

concat_cols = ["nama_usaha", "keberadaan_usaha_label", "kategori", "kbli_label"]


def concat_unique(series, sep=" | "):
    vals = series.dropna().astype(str)
    vals = vals[vals.str.strip() != ""]
    return sep.join(vals.unique())


df_hasil = (
    df_gabungan.groupby(group_cols, dropna=False)
    .agg({col: concat_unique for col in concat_cols})
    .reset_index()
)

print(f"Total baris setelah groupby: {len(df_hasil)}")

In [ ]:
df_hasil.to_excel("../fkp_checker.xlsx")